In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from datetime import datetime
from typing import Any, Dict, List
from sqlalchemy import Integer, JSON, String, Text, func, text
from sqlalchemy.orm import DeclarativeBase, declared_attr, Mapped, mapped_column
from sqlalchemy.ext.asyncio import AsyncAttrs, async_sessionmaker, create_async_engine
from pgvector.sqlalchemy import Vector
import polars
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_postgres import PGVector
from langchain_huggingface import HuggingFaceEmbeddings

c:\main\data_science\projects\dl_practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
NOTEBOOK_DIR = Path.cwd()
dotenv_path = NOTEBOOK_DIR / '.env'
load_dotenv(dotenv_path=str(dotenv_path))

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
GIGACHAT_API_KEY = os.environ.get("GIGACHAT_API_KEY")
HUGGINGFACEHUB_API_TOKEN = os.environ.get("HUGGINGFACEHUB_API_TOKEN")

POSTGRES_DB_NAME = "userdb"

POSTGRES_URL = (
    f"postgresql+psycopg2://user"
    f":user"
    f"@localhost"
    f":5436"
    f"/{POSTGRES_DB_NAME}"
)

In [44]:
root_path = Path.cwd().parent.parent
DATASETS_DIR_PATH = root_path / "resources" / "datasets" / "india-news-headlines.csv"
DATASETS_DIR_PATH

WindowsPath('c:/main/data_science/projects/dl_practice/resources/datasets/india-news-headlines.csv')

## Задание 1: Базовый RAG с LangChain и PGVector

Цель: Научиться интегрировать LangChain с векторной базой данных (PGVector) для классического RAG.
Технологии: LangChain, PGVector, Transformers (Hugging Face), Python.
Шаги:

Установите PGVector и настройте локальную базу данных.
Используйте LangChain для загрузки и чанкинга текстового датасета (например, Википедия или документация).
Создайте эмбеддинги с помощью модели sentence-transformers/all-MiniLM-L6-v2.
Сохраните эмбеддинги в PGVector.
Реализуйте простой ретривер с использованием LangChain и PGVector.
Протестируйте систему на нескольких запросах.
Кейс:

Создайте RAG-систему для поиска информации о исторических событиях (например, "Вторая мировая война").
Оцените качество ответов с помощью метрик (например, точность@k).


In [45]:
df = polars.read_csv(DATASETS_DIR_PATH)
sentences = df[:1000]
sentences

publish_date,headline_category,headline_text
i64,str,str
20010102,"""unknown""","""Status quo will not be disturb…"
20010102,"""unknown""","""Fissures in Hurriyat over Pak …"
20010102,"""unknown""","""America's unwanted heading for…"
20010102,"""unknown""","""For bigwigs; it is destination…"
20010102,"""unknown""","""Extra buses to clear tourist t…"
…,…,…
20010129,"""unknown""","""UAE NRIs mobilise relief for q…"
20010129,"""unknown""","""Tigers refuse to sign Norwegia…"
20010129,"""unknown""","""Fiji Hindus protest religious …"


In [59]:
sentences['headline_category'].value_counts()

headline_category,count
str,u32
"""unknown""",984
"""edit-page""",1
"""city.delhi""",1
"""city.bengaluru""",2
"""entertainment.hindi.bollywood""",2
"""city.patna""",5
"""entertainment.english.hollywoo…",2
"""business.india-business""",1
"""india""",2


In [46]:
# model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
# embeddings = model.encode(sentences)
# print(embeddings)

In [48]:
collection_name = "my_collection"

docs = [Document(
            page_content=sentence['headline_text'],
            metadata={"headline_category": sentence['headline_category'],
                      "publish_date": str(sentence['publish_date'])}
        )
        for sentence in sentences.to_dicts()]

vectorstore = PGVector(
    connection=POSTGRES_URL,
    embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
    collection_name=collection_name,
    use_jsonb=True
)

vectorstore.add_documents(docs)

['2fed377e-badc-48e3-94af-9d14f602cacc',
 'a4e064c9-a87d-4e9b-94a9-9d8cc58c5554',
 '85509782-e859-434c-aac5-100c7d0c827d',
 'b62ec0c6-ef5f-4507-9542-526eb26eea7d',
 'd0060fab-6029-409f-aefe-b42420012646',
 'a59a29c0-29ae-4a9b-88c4-c0145c9123d8',
 '485fc7e0-fec6-4d73-86a9-84b1d619bd68',
 'b1f2ab61-9b0d-4889-9369-c02709c92960',
 'e43d0bc1-017f-486c-842a-262d44c55730',
 '0cc49a40-5e2a-4cdb-9542-76f9a1c4b0b7',
 '43d6f02e-e662-46b8-8f08-44e260c7d57c',
 '07533fd5-4687-408c-9ea0-9dc928ded297',
 'c49cefcc-b281-4277-93a9-90dc961259bc',
 '191c7790-7fe4-42b1-b257-b4e19f438bdd',
 '8d66988e-70b4-4618-a8f9-5187e4a02abd',
 '4ac92514-296d-41cd-a09f-2e7a31d2b54f',
 'c15384e8-6a6d-4986-bf61-41e0bb1f18db',
 '05f21b3d-b3ca-4551-a2bd-0c70b73b83bd',
 '96b76737-4a4f-4b26-8b7f-26b268353a8b',
 '36f49b56-ba74-4dcb-8a34-87b09c20fcca',
 '5e10db50-63bb-4c14-a7b8-59a4bb487d2d',
 'b6711a4b-6c0f-45a0-bd82-3a7dcdf8ce3e',
 '15ca0f95-e1ac-4d26-88d7-05b1180029f5',
 'a04e39a4-a1c1-4e9a-8c22-5b688768b7da',
 'b4ecdc97-e091-

In [51]:
res = vectorstore.similarity_search("Status quo", k=5)

for i, doc in enumerate(res):
    print(f"{i+1}. {doc.page_content}")
    print(f"   Категория: {doc.metadata['headline_category']}")
    print(f"   Дата: {doc.metadata['publish_date']}\n")

1. Status quo will not be disturbed at Ayodhya; says Vajpayee
   Категория: unknown
   Дата: 20010102

2. Status quo will not be disturbed at Ayodhya; says Vajpayee
   Категория: unknown
   Дата: 20010102

3. Processing Peace
   Категория: unknown
   Дата: 20010129

4. Ceasefire decision likely today
   Категория: unknown
   Дата: 20010123

5. Our department is life
   Категория: unknown
   Дата: 20010124



In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

query = "Status quo"
relevant_docs = retriever.invoke(query)
print(relevant_docs)

[Document(id='2fed377e-badc-48e3-94af-9d14f602cacc', metadata={'publish_date': '20010102', 'headline_category': 'unknown'}, page_content='Status quo will not be disturbed at Ayodhya; says Vajpayee'), Document(id='d7447c62-ec89-4324-8450-d120a213fcdd', metadata={'publish_date': '20010102', 'headline_category': 'unknown'}, page_content='Status quo will not be disturbed at Ayodhya; says Vajpayee'), Document(id='e4b2e9d0-5551-4663-9e0b-d5a49901cbb0', metadata={'publish_date': '20010129', 'headline_category': 'unknown'}, page_content='Processing Peace')]


In [56]:
query = "world war 2"
relevant_docs_ww2 = retriever.invoke(query)
print(*relevant_docs_ww2, sep='\n')

page_content='Soccer helps heal the wounds of war' metadata={'publish_date': '20010126', 'headline_category': 'unknown'}
page_content='Conflict Diamonds' metadata={'publish_date': '20010108', 'headline_category': 'unknown'}
page_content='World govt touted as solution to crises' metadata={'publish_date': '20010104', 'headline_category': 'unknown'}


In [67]:
retriever_update = vectorstore.as_retriever(search_kwargs={
    'k': 3,
    'filter': {
        'headline_category': "city.patna",
    },
    "search_type": "similarity_score_threshold",
    "score_threshold": 0.8
})

query = "medicines"
relevant_docs = retriever_update.invoke(query)
print(*relevant_docs, sep="\n")

page_content='Druggists' stir leads to shortage of medicines' metadata={'publish_date': '20010104', 'headline_category': 'city.patna'}
page_content='Fend for yourselves; Pande tells doctors' metadata={'publish_date': '20010110', 'headline_category': 'city.patna'}
page_content='Bureaucracy undermining legislature's 'existence'' metadata={'publish_date': '20010110', 'headline_category': 'city.patna'}


In [76]:
collection_name = "simple_collection"

docs = [
    Document(page_content="Пример текста 1 о ретривере.", metadata={"source": "doc1"}),
    Document(page_content="Пример текста 2 о LangChain.", metadata={"source": "doc2"}),
]

vectorstore = PGVector(
    connection=POSTGRES_URL,
    embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
    collection_name=collection_name,
    use_jsonb=True
)
vectorstore.add_documents(docs)

retriever_simple = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, context_recall
from datasets import Dataset

retrieved_docs = retriever_simple.invoke("Что такое ретривер?")
retrieved_texts = [doc.page_content for doc in retrieved_docs]
retrieved_metadata = [doc.metadata for doc in retrieved_docs]

data = {
    "question": ["Что такое ретривер?"],
    "contexts": [retrieved_texts],
    "answer": ["Ответ на основе ретривера"],
    "ground_truth": ["ground_truth_doc1, ground_truth_doc2"]
}

dataset = Dataset.from_dict(data)
print(dataset)

Dataset({
    features: ['question', 'contexts', 'answer', 'ground_truth'],
    num_rows: 1
})


C:\Users\Tumbi\AppData\Local\Temp\ipykernel_13488\699330622.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall, context_recall
C:\Users\Tumbi\AppData\Local\Temp\ipykernel_13488\699330622.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_recall
  from ragas.metrics import context_precision, context_recall, context_recall


In [85]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_gigachat.chat_models import GigaChat


llm = GigaChat(
    credentials=GIGACHAT_API_KEY,
    verify_ssl_certs=False,
    model="GigaChat:latest",
    scope="GIGACHAT_API_B2B",
    temperature=0.7
)

wrapper_llm = LangchainLLMWrapper(llm)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
wrapper_embeddings = LangchainEmbeddingsWrapper(embeddings)

C:\Users\Tumbi\AppData\Local\Temp\ipykernel_13488\2365591769.py:14: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  wrapper_llm = LangchainLLMWrapper(llm)
C:\Users\Tumbi\AppData\Local\Temp\ipykernel_13488\2365591769.py:17: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  wrapper_embeddings = LangchainEmbeddingsWrapper(embeddings)


In [86]:
result = evaluate(
    dataset,
    metrics=[context_precision, context_recall],
    llm=wrapper_llm,
    embeddings=wrapper_embeddings
)

result

Evaluating: 100%|██████████| 2/2 [00:06<00:00,  3.04s/it]


{'context_precision': 1.0000, 'context_recall': 0.0000}

In [ ]:
def evaluate_retriever_manually(retriever, test_questions, ground_truths):
    results = []
    for question, gt in zip(test_questions, ground_truths):
        retrieved_docs = retriever.invoke(question)
        retrieved_texts = [doc.page_content for doc in retrieved_docs]

        relevant_found = sum(1 for r in retrieved_texts if any(gt_text in r for gt_text in gt))
        precision = relevant_found / len(retrieved_texts)
        
        relevant_total = len(gt)
        recall = relevant_found / relevant_total
        
        results.append({"question": question, "precision": precision, "recall": recall})
    
    return results


test_data = [
    ("Status quo", ["Status quo will not be disturb…"]),
    ("Pakistan", ["Fissures in Hurriyat over Pak …"])
]

results = evaluate_retriever_manually(retriever_simple, 
                                   [q[0] for q in test_data], 
                                   [q[1] for q in test_data])
print(results)


[{'question': 'Status quo', 'precision': 0.0, 'recall': 0.0}, {'question': 'Pakistan', 'precision': 0.0, 'recall': 0.0}]


In [73]:
# Очистка
vectorstore.drop_tables()

## RAGAS

In [3]:
import numpy as np
from langchain_gigachat.chat_models import GigaChat
from langchain_huggingface import HuggingFaceEmbeddings

llm = GigaChat(
    credentials=GIGACHAT_API_KEY,
    verify_ssl_certs=False,
    model="GigaChat:latest",
    scope="GIGACHAT_API_B2B",
    temperature=0.7
)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [4]:
documents = [
    "Saturn is famous for its prominent ring system made of ice particles, rock, and dust.",
    "Mars is often called the Red Planet because of its reddish appearance caused by iron oxide on its surface.",
    "Jupiter is the largest planet in the solar system and is known for its Great Red Spot, a giant storm.",
    "Venus has a thick, toxic atmosphere primarily composed of carbon dioxide, with clouds of sulfuric acid.",
    "Mercury is the closest planet to the Sun and has no atmosphere to retain heat.",
    "Mercury has no atmosphere, is heavily cratered, grey and rocky. It is the smallest planet.",
    "Venus’s atmosphere is thick and toxic, mostly carbon dioxide with sulfuric acid clouds, hottest planet.",
    "Mars is the red planet due to iron oxide (rust) on its surface.",
    "Jupiter is the largest planet, gas giant with Great Red Spot storm.",
    "Saturn has magnificent rings of rock and ice chunks."
]

In [ ]:
class SimpleRAG:
    def __init__(self):
        self.embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
        self.doc_embeddings = self.embedding_model.encode(documents)
        self.documents = documents

    def get_most_relevant_docs(self, query, top_k=2):
        query_emb = self.embedding_model.encode([query])
        similarities = np.dot(self.doc_embeddings, query_emb.T).flatten()
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        return [self.documents[i] for i in top_indices]

    def generate_answer(self, query, retrieved):
        context = "\n".join(retrieved)
        prompt = f"""
        Answer the question based only on the context. If not in context, say "I don't know".

        Question: {query}

        Context: {context}

        Answer:
        """
        
        response = llm.invoke(
            [{"role": "user", "content": prompt}],
            max_tokens=150,
            temperature=0.1
        )
        return response.content.strip()


rag = SimpleRAG()

In [12]:
sample_queries = [
    "Which planet is known for its rings?",
    "Why is Mars called the Red Planet?",
    "What is the largest planet in our solar system?",
    "What is unique about Venus's atmosphere?",
    "Which planet has no atmosphere?"
]

expected_responses = [
    "Saturn is famous for its prominent ring system made of ice particles, rock, and dust.",
    "Mars is often called the Red Planet because of its reddish appearance caused by iron oxide on its surface.",
    "Jupiter is the largest planet in the solar system and is known for its Great Red Spot, a giant storm.",
    "Venus has a thick, toxic atmosphere primarily composed of carbon dioxide, with clouds of sulfuric acid.",
    "Mercury is the closest planet to the Sun and has no atmosphere to retain heat."
]

dataset = []

for query, reference in zip(sample_queries, expected_responses):
    retrieved = rag.get_most_relevant_docs(query)
    response = rag.generate_answer(query, retrieved)

    dataset.append({
        "user_input": query,
        "retrieved_contexts": retrieved,
        "response": response,
        "reference": reference
    })

    print(f"Query: {query}")
    print(f"Retrieved: {retrieved}")
    print(f"Generated: {response}")
    print(f"Reference: {reference}\n")

Query: Which planet is known for its rings?
Retrieved: ['Saturn is famous for its prominent ring system made of ice particles, rock, and dust.', 'Saturn has magnificent rings of rock and ice chunks.']
Generated: Answer: **Saturn**
Reference: Saturn is famous for its prominent ring system made of ice particles, rock, and dust.

Query: Why is Mars called the Red Planet?
Retrieved: ['Mars is often called the Red Planet because of its reddish appearance caused by iron oxide on its surface.', 'Mars is the red planet due to iron oxide (rust) on its surface.']
Generated: Answer: Mars is called the Red Planet because of its distinctive reddish color, which is caused by iron oxide (commonly known as rust) present on its surface.
Reference: Mars is often called the Red Planet because of its reddish appearance caused by iron oxide on its surface.

Query: What is the largest planet in our solar system?
Retrieved: ['Jupiter is the largest planet in the solar system and is known for its Great Red Sp

In [13]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_list(dataset)

In [14]:
from ragas import evaluate

from ragas.llms import LangchainLLMWrapper

from ragas.metrics import (
    LLMContextRecall,
    ContextPrecision,
    Faithfulness,
    FactualCorrectness
)

evaluator_llm = LangchainLLMWrapper(llm)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        LLMContextRecall(),
        ContextPrecision(),
        Faithfulness(),
        FactualCorrectness()
    ],
    llm=evaluator_llm
)

print(result)

C:\Users\Tumbi\AppData\Local\Temp\ipykernel_24824\2872284628.py:5: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\Tumbi\AppData\Local\Temp\ipykernel_24824\2872284628.py:5: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\Tumbi\AppData\Local\Temp\ipykernel_24824\2872284628.py:5: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\Tumbi\AppData\Local\Temp\ipykernel_2482

{'context_recall': 1.0000, 'context_precision': 1.0000, 'faithfulness': 0.8833, 'factual_correctness(mode=f1)': 0.4160}
